In [1]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


Uper Data - EDA 

Data prepareing and cleaning

In [2]:
# Load dataset
df = pd.read_csv('UberDataset.csv')

# Inspect structure
df.info()
df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1156 entries, 0 to 1155
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   START_DATE  1156 non-null   object 
 1   END_DATE    1155 non-null   object 
 2   CATEGORY    1155 non-null   object 
 3   START       1155 non-null   object 
 4   STOP        1155 non-null   object 
 5   MILES       1156 non-null   float64
 6   PURPOSE     653 non-null    object 
dtypes: float64(1), object(6)
memory usage: 63.3+ KB


,START_DATE,END_DATE,CATEGORY,START,STOP,MILES,PURPOSE
0,01-01-2016 21:11,01-01-2016 21:17,Business,Fort Pierce,Fort Pierce,5.1,Meal/Entertain
1,01-02-2016 01:25,01-02-2016 01:37,Business,Fort Pierce,Fort Pierce,5.0,NaN
2,01-02-2016 20:25,01-02-2016 20:38,Business,Fort Pierce,Fort Pierce,4.8,Errand/Supplies
3,01-05-2016 17:31,01-05-2016 17:45,Business,Fort Pierce,Fort Pierce,4.7,Meeting
4,01-06-2016 14:42,01-06-2016 15:49,Business,Fort Pierce,West Palm Beach,63.7,Customer Visit


In [3]:
# Check for duplicates
print(f"Duplicate rows: {df.duplicated().sum()}")

# Check for missing values
print(f"Missing values:\n{df.isnull().sum()}")


Duplicate rows: 1
Missing values:
START_DATE      0
END_DATE        1
CATEGORY        1
START           1
STOP            1
MILES           0
PURPOSE       503
dtype: int64


In [4]:

# drop duplicates
df = df.drop_duplicates()

# missing values
df['PURPOSE'] = df['PURPOSE'].fillna('Unknown')
df = df.dropna()

print(f"Missing values:\n{df.isnull().sum()}")
print(f"Number of rows: {df.shape[0]}")


Missing values:
START_DATE    0
END_DATE      0
CATEGORY      0
START         0
STOP          0
MILES         0
PURPOSE       0
dtype: int64
Number of rows: 1154


In [5]:
# Convert date columns
#df['START_DATE'] = pd.to_datetime(df['START_DATE'], dayfirst=True)
#df['END_DATE'] = pd.to_datetime(df['END_DATE'], errors='coerce')


In [6]:
# date columns seem to have multiple formats
# identifying formats
fingerprints1 = df['START_DATE'].astype(str).str.replace(r'\d', '#', regex=True).str.replace(r'[a-zA-Z]', 'A', regex=True)


print(fingerprints1.value_counts())

fingerprints2 = df['END_DATE'].astype(str).str.replace(r'\d', '#', regex=True).str.replace(r'[a-zA-Z]', 'A', regex=True)

print(fingerprints2.value_counts())

START_DATE
#/##/#### ##:##     426
##-##-#### ##:##    421
##/##/#### ##:##    221
#/##/#### #:##       59
##/##/#### #:##      27
Name: count, dtype: int64
END_DATE
#/##/#### ##:##     427
##-##-#### ##:##    420
##/##/#### ##:##    224
#/##/#### #:##       59
##/##/#### #:##      24
Name: count, dtype: int64


In [7]:
# handling date inconsistencies 

df['start_date'] = pd.to_datetime(df['START_DATE'], format='mixed', dayfirst=False)

#df['START_DATE'].notnull().sum() - df['start_date'].notnull().sum()

df['end_date'] = pd.to_datetime(df['END_DATE'], format='mixed', dayfirst=False)

#df['END_DATE'].notnull().sum() - df['end_date'].notnull().sum()


In [8]:
# extract time features
df['start_day'] = df['start_date'].dt.day_name()
df['start_day'] = pd.Categorical(df['start_day'], categories=['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'])

df['duration'] = (df['end_date'] - df['start_date']).dt.total_seconds() /60

df['hour'] = df['start_date'].dt.hour

def time_of_day(hour):
    if 5 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 21:
        return 'Evening'
    else:
        return 'Night'

df['time_of_day'] = df['hour'].apply(time_of_day)
df['time_of_day'] = pd.Categorical(df['time_of_day'], categories= ['Morning', 'Afternoon', 'Evening', 'Night'])

df.info()



<class 'pandas.core.frame.DataFrame'>
Index: 1154 entries, 0 to 1154
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   START_DATE   1154 non-null   object        
 1   END_DATE     1154 non-null   object        
 2   CATEGORY     1154 non-null   object        
 3   START        1154 non-null   object        
 4   STOP         1154 non-null   object        
 5   MILES        1154 non-null   float64       
 6   PURPOSE      1154 non-null   object        
 7   start_date   1154 non-null   datetime64[ns]
 8   end_date     1154 non-null   datetime64[ns]
 9   start_day    1154 non-null   category      
 10  duration     1154 non-null   float64       
 11  hour         1154 non-null   int32         
 12  time_of_day  1154 non-null   category      
dtypes: category(2), datetime64[ns](2), float64(2), int32(1), object(6)
memory usage: 106.5+ KB


In [9]:
#milage vs duration (sanity check)
px.scatter(
    df,
    x= "duration",
    y = "MILES"
)

Some points are illogical regarding duration of trip and milage. These are usually the result of sensor glitches, app timeouts, or manual entry errors.

In [10]:
# Speed = Distance / (Duration in hours)
df['speed_mph'] = df['MILES'] / (df['duration'] / 60)

# Considering: speed should be > 2 mph and < 85 mph
illogical_trips = df[(df['speed_mph'] > 85) | (df['speed_mph'] < 5) | (df['duration'] >200)]

print(f"Number of illogical trips: {len(illogical_trips)}")
illogical_trips[['START_DATE', 'END_DATE','PURPOSE', 'MILES', 'duration', 'speed_mph']].sort_values("duration")

Number of illogical trips: 28


,START_DATE,END_DATE,PURPOSE,MILES,duration,speed_mph
761,9/16/2016 7:08,9/16/2016 7:08,Unknown,1.6,0.0,inf
751,09-06-2016 17:49,09-06-2016 17:49,Unknown,69.1,0.0,inf
807,10/13/2016 13:02,10/13/2016 13:02,Unknown,0.7,0.0,inf
798,10-08-2016 15:03,10-08-2016 15:03,Unknown,3.6,0.0,inf
786,10-04-2016 12:17,10-04-2016 12:18,Unknown,15.1,1.0,906.000000
754,09-11-2016 21:40,09-11-2016 21:42,Unknown,9.8,2.0,294.000000
375,5/18/2016 13:00,5/18/2016 13:02,Customer Visit,7.6,2.0,228.000000
773,9/27/2016 8:33,9/27/2016 8:35,Unknown,5.8,2.0,174.000000
789,10-06-2016 18:37,10-06-2016 18:39,Unknown,18.4,2.0,552.000000
799,10-08-2016 18:15,10-08-2016 18:18,Unknown,8.0,3.0,160.000000


The zero duration can be cancelled trips. Very long trips could be waiting time. Very long trips where the timer was likely left running accidentally.

The individual cases needs further investigation. for the purpose of this analysis I will drop them

In [11]:
full_df = df


keep = (df['speed_mph'] <= 85) & (df['speed_mph'] >= 5) & (df['duration'] <= 200)

df = df[keep].copy()

full_df.shape[0] - df.shape[0]


28

In [12]:
px.scatter(
    df,
    x= "duration",
    y = "MILES",
    trendline='ols', trendline_color_override='red'
)

### Common Purposes for Uber Trips

In [13]:
purpose_counts = df['PURPOSE'].value_counts().reset_index()
purpose_counts.columns = ['PURPOSE', 'COUNT']

fig_purpose = px.bar(
    purpose_counts,
    x='PURPOSE',
    y='COUNT',
    title='Most Common Purposes for Uber Trips',
    color='COUNT'
)

fig_purpose.show()

Most of the trip purposes are not known

### Common time of the day 

In [14]:
time_trend= df["time_of_day"].value_counts().reset_index()
time_trend.columns = ['Time of day', 'Count']
fig_time = px.bar(
    time_trend,
    x= 'Time of day',
    y= 'Count'
    
)

fig_time

Most rides are commuted during the afternoon.

### Milage trend 

In [15]:
mileage_trends = df.groupby('hour')['MILES'].mean().reset_index()

mileage_fig = px.line(
    mileage_trends,
    x= 'hour',
    y= 'MILES',
    title = " Trip milage (average) trend over time of the day",
)

mileage_fig.update_xaxes(nticks=24)

rects = [
    (0, 5,  'purple'),   # Night part 1
    (5, 12, 'red'),  # Morning
    (12, 17, 'green'),  # Afternoon
    (17, 21, "#597194"),  # Evening
    (21, 24, 'purple')  # Night part 2
]

for start, end, color in rects:
    mileage_fig.add_vrect(
        x0=start, x1=end, 
        fillcolor=color, opacity=0.3, 
        layer="below", line_width=0
    )

mileage_fig

2 am has the highest average miles travelled

In [16]:
# milage vs umber ot trips

mileage_counts = df.groupby('hour')['MILES'].agg(['mean', 'count']).reset_index()

fig = make_subplots(specs=[[{"secondary_y": True}]])

#Milage (Primary y Axis)
fig.add_trace(
    go.Scatter(
        x=mileage_counts['hour'], 
        y=mileage_counts['mean'], 
        name="Avg Mileage", 
        line=dict(color='darkblue', width=3)
    ),
    secondary_y=False,
)

#Add Number of Trips (Secondary Axis)
fig.add_trace(
    go.Bar(
        x=mileage_counts['hour'], 
        y=mileage_counts['count'], 
        name="Number of Trips", 
        marker_color='gray', 
        opacity=0.4
    ),
    secondary_y=True,
)

#Add your background color time of the day
rects = [
    (0, 5,  'purple'),   # Night part 1
    (5, 12, 'red'),      # Morning
    (12, 17, 'green'),   # Afternoon
    (17, 21, "#597194"), # Evening
    (21, 24, 'purple')   # Night part 2
]

for start, end, color in rects:
    fig.add_vrect(
        x0=start, x1=end, 
        fillcolor=color, opacity=0.15, # Lower opacity so bars are visible
        layer="below", line_width=0
    )

fig.update_layout(
    title_text="Trip Mileage vs. Volume Over Time of Day",
    xaxis_title="Hour of Day",
    template='plotly_white'
)

fig.update_yaxes(title_text="<b>Average</b> Miles", secondary_y=False)
fig.update_yaxes(title_text="<b>Total</b> Number of Trips", secondary_y=True)
fig.update_xaxes(dtick=1)

fig.show()

2 am has the highest average miles although least number of trips

In [17]:
# 2 am trips talble

trips_2am = df[df['hour'] == 2]
trips_2am.sort_values(by='MILES', ascending=False)

,START_DATE,END_DATE,CATEGORY,START,STOP,MILES,PURPOSE,start_date,end_date,start_day,duration,hour,time_of_day,speed_mph
299,04-03-2016 02:00,04-03-2016 04:16,Business,Florence,Cary,159.3,Meeting,2016-04-03 02:00:00,2016-04-03 04:16:00,Sunday,136.0,2,Night,70.279412
455,6/19/2016 2:39,6/19/2016 2:50,Business,Cary,Raleigh,6.0,Unknown,2016-06-19 02:39:00,2016-06-19 02:50:00,Sunday,11.0,2,Night,32.727273


In [18]:
mileage_trends = df.groupby(['PURPOSE', 'time_of_day'])['MILES'].mean().reset_index()

fig_mileage_purpose = px.bar(
    mileage_trends,
    x='MILES',
    y='PURPOSE',
    color='PURPOSE',
    facet_col='time_of_day',

    title='Average Mileage by Purpose and Time of Day'
)

fig_mileage_purpose.show()

C:\Users\Fatema\AppData\Local\Temp\ipykernel_11520\1793100679.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [19]:

fig_mileage = px.bar(
    mileage_trends,
    x='PURPOSE',
    y='MILES',
    color='time_of_day',
    barmode='group',
    title='Average Mileage by Purpose and Time of Day'
)

fig_mileage.show()

In [20]:
# trips with purpose commute

df[df['PURPOSE'] == "Commute"]

,START_DATE,END_DATE,CATEGORY,START,STOP,MILES,PURPOSE,start_date,end_date,start_day,duration,hour,time_of_day,speed_mph
559,7/17/2016 12:20,7/17/2016 15:25,Personal,Boone,Cary,180.2,Commute,2016-07-17 12:20:00,2016-07-17 15:25:00,Sunday,185.0,12,Afternoon,58.443243


For most of the day (Morning and Afternoon), there is a positive correlation: as the number of trips increases, the average mileage remains stable, showing a reliable routine.

The peak at 2 AM, the data is driven by an "exceptional" event rather than a routine. Similarly the millage for the commute purpose.



### Comparative Overview of Trip Lengths


In [21]:
px.box(
    df,
    y= 'MILES'

)

In [22]:
df['CATEGORY'].value_counts()

CATEGORY
Business    1049
Personal      77
Name: count, dtype: int64

In [23]:
fig_length = px.box(
    df,
    y='CATEGORY',
    x='MILES',
    color='CATEGORY',
    title='Trip Length Distribution: Business vs Personal'
)

fig_length.add_trace(
    go.Box(
        x=df['MILES'],
        name='Total (All)',
        marker_color='gray'
    )
)
fig_length.update_layout(showlegend = False)

In [24]:
df.groupby('CATEGORY')['MILES'].describe()

,count,mean,std,min,25%,50%,75%,max
CATEGORY,,,,,,,,
Business,1049.0,9.831173,17.711137,0.5,2.9,6.0,10.4,201.0
Personal,77.0,9.320779,21.220995,0.7,1.9,4.2,8.7,180.2


The portion of category "personal" is small compared to business in this dataset, this explains the resemplance of the milage distribution in "business" compared to the total 

Outliers should be examined for data entry errors or exceptional travel events.

------------------------------


### Dashboard

In [ ]:
import dash
from dash import dcc, html, Input, Output
import dash_bootstrap_components as dbc

In [60]:

app = dash.Dash(__name__, external_stylesheets=[dbc.themes.MORPH])

SIDEBAR_STYLE = {
    "position": "fixed", "top": 0, "left": 0, "bottom": 0,
    "width": "18rem", "padding": "2rem 1rem", "background-color": "#111",
    "color": "white"
}

CONTENT_STYLE = {"margin-left": "20rem", "margin-right": "2rem", "padding": "2rem 1rem"}

app.layout = html.Div([
    # Sidebar
    html.Div([
        html.H2("Uber Portfolio", className="display-6 text-info"),
        html.Hr(style={"color": "white"}),
        html.P("Step into the Data", className="lead text-white-50"),
        dbc.Card([
            dbc.CardBody([
                html.Label("Filter by Purpose:", className="fw-bold mb-2 text-info"),
                dcc.Checklist(
                    id='purpose-filter',
                    options=[{'label': f" {i}", 'value': i} for i in df['PURPOSE'].unique()],
                    value=df['PURPOSE'].unique().tolist(),
                    labelStyle={'display': 'block', 'margin-bottom': '5px'}
                ),
            ])
        ], color='light', outline=True)
    ], style=SIDEBAR_STYLE),

    # Main Content
    html.Div([
        dbc.Container([
            dbc.Row([
                dbc.Col(html.H1("Uber Analytics Dashboard", className="text-center mt-4 mb-4"), width=12),
            ]),

            # KPI Row
            dbc.Row([
                dbc.Col(dbc.Card([html.H5("Avg Miles"), html.H2(id="avg-miles-kpi")], body=True, color="dark", outline=True), width=4),
                dbc.Col(dbc.Card([html.H5("Total Trips"), html.H2(id="total-trips-kpi")], body=True, color="dark", outline=True), width=4),
                dbc.Col(dbc.Card([html.H5("Avg Duration"), html.H2(id="avg-time-kpi")], body=True, color="dark", outline=True), width=4),
            ], className="mb-4 text-center"),

            # Main Visuals Area
            dbc.Row([
                dbc.Col(dcc.Graph(id='mileage-vs-volume-plot'), width=12, className="mb-4"),
                dbc.Col(dcc.Graph(id='mileage-trends-plot'), width=12, className="mb-4"),
                dbc.Col(dcc.Graph(id='category-donut'), width=6),
                dbc.Col(dcc.Graph(id='time-of-day-bar'), width=6),
            ])
        ], fluid=True)
    ], style=CONTENT_STYLE)
])

# Callbacks
@app.callback(
    [Output("avg-miles-kpi", "children"),
     Output("total-trips-kpi", "children"),
     Output("avg-time-kpi", "children"),
     Output("category-donut", "figure"),
     Output("time-of-day-bar", "figure"),
     Output("mileage-trends-plot", "figure"),
     Output("mileage-vs-volume-plot", "figure")],
    Input("purpose-filter", "value")
)
def update_dashboard(selected_purposes):
    filtered_df = df[df['PURPOSE'].isin(selected_purposes)]
    
    # fig KPIs
    avg_miles = f"{filtered_df['MILES'].mean():.1f}"
    total_trips = f"{len(filtered_df):,}"
    avg_time = f"{filtered_df['duration'].mean():.1f}" if 'duration' in filtered_df.columns else "N/A"

    # fig Donut 
    fig_donut = px.pie(filtered_df, names='CATEGORY', hole=.5, template="plotly_dark")

    # fig histo
    fig_bar = px.histogram(filtered_df, x="time_of_day", template="plotly_dark", color_discrete_sequence=['#00cfec'])
    
    # fig Faceted Mileage
    mileage_trends = filtered_df.groupby(['PURPOSE', 'time_of_day'])['MILES'].mean().reset_index()
    fig_mileage_purpose = px.bar(mileage_trends, x='MILES', y='PURPOSE', color='PURPOSE', 
                                 facet_col='time_of_day', template="plotly_dark")

    # THE MILEAGE VS VOLUME
    mileage_counts = filtered_df.groupby('hour')['MILES'].agg(['mean', 'count']).reset_index()
    fig_volume = make_subplots(specs=[[{"secondary_y": True}]])

    # Mileage Line
    fig_volume.add_trace(go.Scatter(
        x=mileage_counts['hour'], y=mileage_counts['mean'], 
        name="Avg Mileage", line=dict(color='#00cfec', width=4)
    ), secondary_y=False)

    # Trip Bar
    fig_volume.add_trace(go.Bar(
        x=mileage_counts['hour'], y=mileage_counts['count'], 
        name="Number of Trips", marker_color='white', opacity=0.3
    ), secondary_y=True)

    # Background Colors for Time of Day
    rects = [(0, 5, 'purple'), (5, 12, 'red'), (12, 17, 'green'), (17, 21, "#597194"), (21, 24, 'purple')]
    for start, end, color in rects:
        fig_volume.add_vrect(x0=start, x1=end, fillcolor=color, opacity=0.5, layer="below", line_width=0)

    fig_volume.update_layout(
        title_text="Mileage vs. Volume Patterns",
        template='plotly_dark',
        plot_bgcolor='rgba(0,0,0,0)',
        showlegend=False
    )
    fig_volume.update_xaxes(title_text="Hour of Day", dtick=1)
    fig_volume.update_yaxes(title_text="Avg Miles", secondary_y=False)
    fig_volume.update_yaxes(title_text="Total Trips", secondary_y=True)

    return avg_miles, total_trips, avg_time, fig_donut, fig_bar, fig_mileage_purpose, fig_volume

if __name__ == "__main__":
    app.run(debug=True, port=8054)

C:\Users\Fatema\AppData\Local\Temp\ipykernel_11520\2104496896.py:81: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

C:\Users\Fatema\AppData\Local\Temp\ipykernel_11520\2104496896.py:81: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

C:\Users\Fatema\AppData\Local\Temp\ipykernel_11520\2104496896.py:81: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

C:\Users\Fatema\AppData\Local\Temp\ipykernel_11520\2104496896.py:81: FutureWarn

localhost http://127.0.0.1:8054/

In [ ]:
!jupyter nbconvert --to html EDA-UBER.ipynb
 

[NbConvertApp] Converting notebook EDA-UBER.ipynb to html
C:\Users\Fatema\AppData\Local\Programs\Python\Python311\share\jupyter\nbconvert\templates\base\display_priority.j2:32: UserWarning: Your element with mimetype(s) dict_keys(['application/vnd.plotly.v1+json']) is not able to be represented.
  {%- elif type == 'application/javascript' -%}
[NbConvertApp] Writing 676817 bytes to EDA-UBER.html
